# Titanic Survival Prediction

## Introduction

This project uses the Titanic Survival Dataset to build a classification model that predicts whether a passenger survived the sinking of the Titanic, based on passenger attributes.

The model uses cross-validation and hyperparameter grid search to optimize the machine learning pipeline. We'll build a Random Forest Classifier first, then modify the pipeline to use Logistic Regression and compare the results.

### Install required libraries

### Import required libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

### Titanic Passenger Dataset

We'll be working with the Titanic passenger dataset to build a classification model to predict whether a passenger survived the sinking of the Titanic.

**Data Dictionary:**

| Variable   |	Definition   |
 |:------|:--------------------------------|
 |survived | survived? 0 = No, 1 = yes  |
 |pclass | Ticket class (int)  |
 |sex	 |sex |
 |age	 | age in years  |
 |sibsp  |	# of siblings / spouses aboard the Titanic |
 |parch  |	# of parents / children aboard the Titanic |
 |fare   |	Passenger fare   |
 |embarked | Port of Embarkation |
 |class  |Ticket class (obj)   |
 |who    | man, woman, or child  |
 |adult_male | True/False |
 |alive  | yes/no  |
 |alone  | yes/no  |

## Load the Titanic dataset using Seaborn


In [ ]:
titanic = sns.load_dataset('titanic')
titanic.head()

### Select relevant features and the target


In [ ]:
titanic.count()

**Features to drop:**
- `deck` has a lot of missing values so we'll drop it
- `age` has quite a few missing values as well
- `embarked` and `embark_town` don't seem relevant so we'll drop them
- `alive` is unclear so we'll ignore it

**Target:**
- `survived` is our target class variable

In [ ]:
features = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'class', 'who', 'adult_male', 'alone']
target = 'survived'

X = titanic[features]
y = titanic[target]

### Check class balance

In [ ]:
y.value_counts()

About 38% of the passengers in the dataset survived. Due to this slight imbalance, we'll stratify the data when performing train/test split and for cross-validation.

### Split the data into training and testing sets

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

### Define preprocessing transformers for numerical and categorical features

Automatically detect numerical and categorical columns and assign them to separate numeric and categorical features.

In [ ]:
numerical_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category', 'string']).columns.tolist()

Define separate preprocessing pipelines for both feature types.

In [ ]:
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

Combine the transformers into a single column transformer. We'll use the sklearn "column transformer" estimator to separately transform the features, which will then concatenate the output as a single feature space, ready for input to a machine learning estimator.

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

### Create a model pipeline

Complete the model pipeline by combining the preprocessing with a Random Forest classifier.

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

### Define a parameter grid

We'll use the grid in a cross-validation search to optimize the model.

In [ ]:
param_grid = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5]
}

### Perform grid search cross-validation and fit the best model to the training data

In [ ]:
# Cross-validation method
cv = StratifiedKFold(n_splits=5, shuffle=True)

### Train the pipeline model

In [ ]:
model = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=cv, scoring='accuracy', verbose=2)
model.fit(X_train, y_train)

### Get model predictions and classification report

In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

### Plot the confusion matrix

In [ ]:
conf_matrix = confusion_matrix(y_test, y_pred)

plt.figure()
sns.heatmap(conf_matrix, annot=True, cmap='Blues', fmt='d')

# Set the title and labels
plt.title('Titanic Classification Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')

# Show the plot
plt.tight_layout()
plt.show()

## Feature Importances

Let's figure out how to get the feature importances of our overall model. First, to obtain the categorical feature importances, we have to work our way backward through the modelling pipeline to associate the feature importances with their one-hot encoded input features that were transformed from the original categorical features.

We don't need to trace back through the pipeline for the numerical features, because we didn't transform them into new ones in any way. Remember, we went from categorical features to one-hot encoded features, using the 'cat' column transformer.

Here's how you trace back through the trained model to access the one-hot encoded feature names:

In [ ]:
model.best_estimator_['preprocessor'].named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features)

Notice how the one-hot encoded features are named - for example, `sex` was split into two boolean features indicating whether the sex is male or female.

Now let's get all of the feature importances and associate them with their transformed feature names.

In [ ]:
feature_importances = model.best_estimator_['classifier'].feature_importances_

# Combine the numerical and one-hot encoded categorical feature names
feature_names = numerical_features + list(model.best_estimator_['preprocessor']
                                        .named_transformers_['cat']
                                        .named_steps['onehot']
                                        .get_feature_names_out(categorical_features))

### Display the feature importances in a bar plot

In [ ]:
importance_df = pd.DataFrame({'Feature': feature_names,
                              'Importance': feature_importances
                             }).sort_values(by='Importance', ascending=False)

# Plotting
plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='skyblue')
plt.gca().invert_yaxis() 
plt.title('Most Important Features in predicting whether a passenger survived')
plt.xlabel('Importance Score')
plt.show()

# Print test score 
test_score = model.score(X_test, y_test)
print(f"\nTest set accuracy: {test_score:.2%}")

### Analysis of feature importances

The test set accuracy is somewhat satisfactory. However, regarding the feature importances, it's crucially important to realize that there is most likely plenty of dependence amongst these variables, and a more detailed modelling approach including correlation analysis is required to draw proper conclusions. For example, there is significant information shared by the variables `age`, `sex_male`, and `who_man`.

## Try Another Model

In practice you would want to try out different models and even revisit the data analysis to improve your model performance. Maybe you can engineer new features or impute missing values to be able to use more data.

With Scikit-learn's powerful pipeline class, this is easy to do in a few steps. Let's update the pipeline and the parameter grid so we can train a Logistic Regression model and compare the performance of the two models.

In [ ]:
### Update the pipeline with Logistic Regression

In [ ]:
pipeline.set_params(classifier=LogisticRegression(max_iter=1000))

### Update the parameter grid for Logistic Regression

In [ ]:
param_grid = {
    'preprocessor__num__imputer__strategy': ['mean', 'median'],
    'classifier__C': [0.1, 1.0, 10.0]
}

### Train the Logistic Regression model

In [ ]:
model = GridSearchCV(estimator=pipeline, param_grid=param_grid, cv=cv, scoring='accuracy', verbose=2)
model.fit(X_train, y_train)

### Get predictions and classification report for Logistic Regression

In [ ]:
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))

### Plot the confusion matrix for Logistic Regression

In [ ]:
conf_matrix = confusion_matrix(y_test, y_pred)

plt.figure()
sns.heatmap(conf_matrix, annot=True, cmap='Blues', fmt='d')

# Set the title and labels
plt.title('Titanic Classification Confusion Matrix (Logistic Regression)')
plt.xlabel('Predicted')
plt.ylabel('Actual')

# Show the plot
plt.tight_layout()
plt.show()

### Get feature coefficients for Logistic Regression

In [ ]:
# Get the feature names after preprocessing
feature_names = numerical_features + list(model.best_estimator_.named_steps['preprocessor'].named_transformers_['cat'].named_steps['onehot'].get_feature_names_out(categorical_features))

# Get the coefficients
coefficients = model.best_estimator_.named_steps['classifier'].coef_[0]

# Create a DataFrame for easier visualization
coef_df = pd.DataFrame({'feature': feature_names, 'coefficient': coefficients})
coef_df = coef_df.sort_values('coefficient', ascending=False)

print(coef_df)

### Display the feature coefficients in a bar plot

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=coef_df, x='coefficient', y='feature', palette='viridis')
plt.title('Feature Coefficients (Logistic Regression)')
plt.xlabel('Coefficient Value')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

### Model Comparison

Comparing the two models:
- **Random Forest**: Achieved good accuracy with feature importances highlighting `sex_male`, `who_man`, and `class_Third` as important predictors.
- **Logistic Regression**: Achieved comparable accuracy with coefficients showing similar patterns - negative coefficients for male and third-class passengers indicate lower survival probability.

Both models identify similar key features, which validates the importance of passenger demographics and class in survival prediction.